# Script to parse jedi files

In [ ]:
import glob
import numpy as np
import pandas as pd
import datetime as dt

import matplotlib.pyplot as plt 
import matplotlib.colors as colors
from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection

from JunoPy.jedi import JediChannel, JediData
from JunoPy.index import IndexFile

In [ ]:
startevent = dt.datetime.strptime('2017-236T20:21:15', '%Y-%jT%H:%M:%S')
endevent   = dt.datetime.strptime('2017-236T20:21:38', '%Y-%jT%H:%M:%S')
start = startevent - dt.timedelta(minutes=10)
end   = endevent + dt.timedelta(minutes=10)

# jedifiles = glob.glob('data/jedi/*SISP_CDR_{:s}_*.TAB'.format(start.strftime('%Y%j')))
# if not jedifiles:
jediindex = IndexFile(dataset='jedi')
jediindex.get_files_between_dates(
    start, end, DoDownload=True, sid='ERSISP')
jediindex.get_files_between_dates(
    start, end, DoDownload=True, sid='ERSESP')
    
jedifiles = glob.glob('data/jedi/*_CDR_{:s}_*.TAB'.format(start.strftime('%Y%j')))
jedidatas = []
for jedifile in jedifiles:
    jedidatas.append(JediData(filename=jedifile))
    print(jedifile)
    


In [ ]:
jedidata = jedidatas[3]
jedidata.df = jedidata.df.loc[start:end]
print([x for x in zip(jedidata.FluxChannels.lowE, jedidata.FluxChannels.highE)])

In [ ]:
print([k for k in jedidata.df.keys()])

##  Create the spectrogram data we need
X - time  
Y - frequency bins  
C - flux added from each look direction  

In [ ]:
def plot_spectrogram(data):
    """
    Plot the energy versus time spectra and combine data from all telescopes.
    """
    # First, we need to create a single array for the energy levels 
    # nSSD is always 6
    # nEnergy levels varies
    
    n_bins = int(len(data.FluxChannels.lowE) / data.nSSD)
    energy_bins_lower = np.zeros(n_bins)
    energy_bins_upper = np.zeros(n_bins)
    lowE = np.asarray([float(x) for x in data.FluxChannels.lowE])
    highE = np.asarray([float(x) for x in data.FluxChannels.highE])
    
    
    print(data.FluxChannels.columns)
    print(data.FluxChannels.typeparticle)
    for j in range(n_bins):
        energy_bins_lower[j] = np.min(lowE[j: j + int((data.nSSD - 1) * n_bins): n_bins])
        energy_bins_upper[j] = np.max(highE[j: j + int((data.nSSD - 1) * n_bins): n_bins])
    
    
    # Combine fluxes from different telescopes.
    for i in range(n_bins):
        channels = [x for x in data.FluxChannels.columns if 'XC{:02d}'.format(i) in x]
        col = data.df.mask(df[channels] > 10**5, 0)
        # .sum()
        data.df['FluxNet_bin{:d}'.format(i)] = col
    # print(data.df['FluxNet_bin4'])
    
    
plot_spectrogram(jedidata)
    
    

In [ ]:


df = jedidata.df
df['endtime'] = df.apply(lambda x: x.name + dt.timedelta(seconds=x['Duration']), axis=1)

x = np.zeros(len(df) + 1, dtype=object)
x[:-1] = df.index
x[-1] = df['endtime'][-1]
    
fig, ax = plt.subplots(6, 1, sharex=True, figsize=(11, 8.5), gridspec_kw=dict(hspace=0))

for iax, a in enumerate(ax):

    columns = [x for x in jedidata.FluxChannels.columns if 'T{:d}'.format(iax) in x]
    jedidata.plot_flux_spectra(a, iax)
        
plt.show()

## 3. Plot the pitch angle spectrogram. 

In [ ]:

# Calculate the pitch angle - 
# Integrate the energy flux in the channel. 

df = jedidata.df
df['endtime'] = df.apply(lambda x: x.name + dt.timedelta(seconds=x['Duration']), axis=1)

def calc_broadband_flux(x, lowE, upE):
    sum = 0
    for i, val in enumerate(x.values):
        if val > 0:
            sum += val * (float(upE[i]) - float(lowE[i]))
    return sum
    
for iax in range(6):
    columns = [x for x in jedidata.FluxChannels.columns if 'T{:d}'.format(iax) in x]
    freq_lower = jedidata.FluxChannels.lowE[iax * len(columns): (iax + 1) * len(columns)]
    freq_upper = jedidata.FluxChannels.highE[iax * len(columns): (iax + 1) * len(columns)]

    df['BroadbandFlux_T{:d}'.format(iax)] = df.apply(
        lambda x: calc_broadband_flux(
            x[columns], freq_lower, freq_upper), axis=1)


        

In [ ]:

for iax in range(6):
    plt.plot(df.index, df['BroadbandFlux_T{:d}'.format(iax)], '.')

In [ ]:
# df.plot(y=['Pitch_Angle_T{:d}'.format(iax) for iax in range(6)], figsize=(11,8.5), legend=True, markersize=11., style='.')
x_pa = np.array([], dtype='datetime64')
y_pa = np.array([])

print(type(df.index))
for iax in range(6):
    
    x_pa = np.concatenate((x_pa, df.index))
    y_pa = np.concatenate((y_pa, df['Pitch_Angle_T{:d}'.format(iax)].to_numpy()))

# x_pa = dt.datetime(x_pa)
print(type(x_pa[0]))
c = pd.to_datetime(x_pa, unit='ns')
print(c)
plt.hist2d(c.values, y_pa, bins=[50, 12], cmin=0.5, cmap='jet')
plt.colorbar()
plt.show()

## Multiple JEDI files (090-180-270) and making the pitch angle versus time spectra. 

In [ ]:
start = dt.datetime.strptime('2019-089T20:35:40', '%Y-%jT%H:%M:%S') - dt.timedelta(minutes=5)
end   = dt.datetime.strptime('2019-089T20:36:36', '%Y-%jT%H:%M:%S') + dt.timedelta(minutes=5)

jedifiles = glob.glob('data/jedi/*_{:s}_*.TAB'.format(start.strftime('%Y%j')))

